# Data transformation and storage

Load `data/interim/dc_chargers_augmented.csv` into `data/processed/ev_chargers.duckdb`.
All loading code stays in this notebook; [schema.sql](../sql/schema.sql) defines the database.

## Database design

### Relationships

```mermaid
erDiagram
    operator o|--o{ charger : operator_id
    sa4_region o|--o{ charger : sa4_code
    external_station o|--o{ charger : external_station_id
    external_station ||--o{ external_station_connector : has
    connector_type ||--o{ external_station_connector : classifies
    external_station ||--|| station_pricing : snapshot_price
```

Each charger references at most one operator, one SA4 region and one external station; those
foreign keys can be NULL. An external station can serve several charger records and report
several connector types. The load creates one pricing observation per external station,
including an explicit status when its price is missing.

### Column summary

PK = primary key; FK = foreign key.

| Table | Keys | Column summary |
|---|---|---|
| `operator` | `operator_id` (PK) | Unique cleaned `operator_name`. |
| `sa4_region` | `sa4_code` (PK) | `sa4_name`, `state_name`, `boundary_year`, boundary `geom`. |
| `charger` | `charger_record_id` (PK); `operator_id`, `sa4_code`, `external_station_id` (FKs) | Station/address, LGA/postcode, coordinates and point geometry, plug count, rating text/kW, match evidence, source/duplicate flags, and original CSV fields in `raw_record` JSON. |
| `external_station` | `external_station_id` (PK); unique provider/ID pair | `source`, `source_station_id`, `station_name`, `operator_label`, `reported_bays`, `max_power_kw`, `last_verified_utc`, coordinates and point geometry. |
| `connector_type` | `connector_id` (PK) | Unique canonical `connector_name`. |
| `external_station_connector` | Both FKs form the PK | `external_station_id`, `connector_id`: one reported station–connector pair. |
| `station_pricing` | `external_station_id` (PK/FK) | Original `raw_cost`, numeric `amount`, `currency`, `unit`, `parse_status`, and `currency_assumed`. |

**Design justification:** operators, regions, external stations and connectors are normalized to
avoid repeating shared details and to make connector and price queries straightforward. Matching
evidence remains on each charger because match quality differs between source records. This adds
a few joins, exposed by `charger_analysis` (one row per charger) and `charger_connector` (one row
per charger–connector pair). The original CSV JSON deliberately duplicates evidence for audit.

A charger record can describe several plugs or share a physical site with another record.
Mixed ratings retain their text and a NULL scalar kW value. External operator/bay/power values
remain source claims; canonical operator names are not overwritten. OCM/OSM connector aliases
are aligned, and OSM metadata tokens such as `type2_combo:output` are excluded from connector
relationships. Raw fields retain these tokens. Match ambiguity flags describe OCM candidates only.

SA4 boundaries are transformed from EPSG:7844 to EPSG:4326; charger coordinates are assumed to
use EPSG:4326, longitude first. An SA4 key is assigned only for exactly one spatial match, and
non-spatial ABS regions retain NULL geometry. The 2026 boundary vintage differs from the charger
snapshot. R-tree indexes support eligible spatial filters.
[DuckDB spatial documentation](https://duckdb.org/docs/stable/core_extensions/spatial/overview)


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
CSV_PATH = ROOT / "data/interim/dc_chargers_augmented.csv"
SA4_PATH = ROOT / "data/raw/sa4_shapefile/SA4_2026_AUST_GDA2020.shp"
DDL_PATH = ROOT / "sql/schema.sql"
DB_PATH = ROOT / "data/processed/ev_chargers.duckdb"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
EXTENSIONS = DB_PATH.parent / ".duckdb_extensions"
EXTENSIONS.mkdir(exist_ok=True)
CONFIG = {"extension_directory": str(EXTENSIONS.resolve())}

assert "7844" in SA4_PATH.with_suffix(".prj").read_text()
print(f"Input: {CSV_PATH}")

Input: d:\Data Engineering\comp5339-a1\data\interim\dc_chargers_augmented.csv


## Connector and price interpretation

The helper parses only complete, unconditional single rates, such as `$0.55/kWh` or `55c/kWh`.
Cents are converted to dollars. `$` and `c` assume AUD for this Australian dataset, explicitly
flagged by `currency_assumed`; an explicit `AUD` does not require that assumption. Literal
`0.60c/kWh` remains AUD 0.006/kWh, even if the source may contain a typo.

Exact free-charging labels receive status `free` and amount zero without an invented unit.
`fee_unpriced` means a fee is reported without an amount. Empty values are `missing`.
Ranges, discounts, free allowances, mixed tariffs and numbers without units remain `unparsed`.
Every original price description is retained; parsed prices are source claims, not verified
current tariffs. No membership or conditional price is inferred.


In [2]:
import re
from decimal import Decimal


def parse_price(text):
    """Parse only unconditional rates; retain original text separately."""
    value = '' if text is None else str(text).strip()
    if not value:
        return None, None, None, 'missing', False
    if value.casefold() in {'free', 'free (osm)'}:
        return Decimal('0'), None, None, 'free', False
    if value.casefold() == 'fee (osm, unpriced)':
        return None, None, None, 'fee_unpriced', False
    match = re.fullmatch(
        r'(?:(?P<currency>AUD|\$)\s*(?P<dollars>\d+(?:\.\d+)?)|'
        r'(?P<cents>\d+(?:\.\d+)?)\s*c)\s*(?:/|per)\s*'
        r'(?P<unit>kwh|min(?:ute)?s?|h(?:r|our)?s?)', value, re.I)
    if not match:
        return None, None, None, 'unparsed', False
    amount = Decimal(match.group('dollars') or match.group('cents'))
    if match.group('cents') is not None:
        amount /= 100
    unit = match.group('unit').casefold()
    unit = 'kWh' if unit == 'kwh' else ('minute' if unit.startswith('min') else 'hour')
    assumed = (match.group('currency') or '').upper() != 'AUD'
    return amount, 'AUD', unit, 'unit_rate', assumed

CONNECTOR_ALIASES = {
    "type2_combo": "CCS (Type 2)", "chademo": "CHAdeMO",
    "type2": "Type 2 (Socket Only)", "type2_cable": "Type 2 (Tethered Connector)",
}


## Load the data

Read the CSV directly with DuckDB, insert distinct external stations, then load connector pairs,
pricing observations and chargers. Conflicting details for a shared ID fail a key constraint.
A fresh temporary database replaces the output after a successful load, preventing duplicate rows
on rerun. Close existing connections first. The first spatial installation needs internet access.


In [3]:
assert not Path(str(DB_PATH) + ".wal").exists(), "Close/checkpoint the existing database first."
with TemporaryDirectory(dir=DB_PATH.parent) as folder:
    temporary_db = Path(folder) / "chargers.duckdb"
    with duckdb.connect(str(temporary_db), config=CONFIG) as con:
        con.execute(DDL_PATH.read_text())
        con.register("csv_data", con.read_csv(str(CSV_PATH), all_varchar=True, na_values=[]))
        input_rows = con.execute("SELECT count(*) FROM csv_data").fetchone()[0]
        assert input_rows > 0
        con.execute("""
            CREATE TEMP VIEW staged AS
            SELECT d.*, to_json(d) AS source_row,
                   CASE match_source
                     WHEN 'ocm' THEN 'ocm:' || (NULLIF(ocm_id, '')::BIGINT)::VARCHAR
                     WHEN 'osm' THEN 'osm:' || (NULLIF(osm_id, '')::BIGINT)::VARCHAR END
                   AS external_station_id
            FROM csv_data d;
            INSERT INTO operator
            SELECT row_number() OVER (ORDER BY Operator), Operator
            FROM (SELECT DISTINCT Operator FROM staged WHERE Operator <> '');
        """)
        con.execute("""
            INSERT INTO sa4_region
            SELECT SA4_CODE26, SA4_NAME26, STE_NAME26, 2026,
                   CASE WHEN geom IS NULL OR ST_IsEmpty(geom) THEN NULL
                        ELSE ST_Transform(geom, 'EPSG:7844', 'EPSG:4326', always_xy := true) END
            FROM ST_Read(?);
        """, [str(SA4_PATH.resolve())])
        con.execute("""
            INSERT INTO external_station
            SELECT DISTINCT external_station_id, match_source,
                   split_part(external_station_id, ':', 2),
                   NULLIF(CASE match_source WHEN 'ocm' THEN ocm_title ELSE osm_name END, ''),
                   NULLIF(aug_operator, ''), NULLIF(aug_bays, ''),
                   CASE WHEN match_source = 'ocm' THEN NULLIF(ocm_max_power_kw, '')::DOUBLE END,
                   CASE WHEN match_source = 'ocm' THEN NULLIF(ocm_date_last_verified, '')::TIMESTAMPTZ AT TIME ZONE 'UTC' END,
                   (CASE match_source WHEN 'ocm' THEN ocm_latitude ELSE osm_latitude END)::DOUBLE AS lat,
                   (CASE match_source WHEN 'ocm' THEN ocm_longitude ELSE osm_longitude END)::DOUBLE AS lon,
                   ST_Point(lon, lat)
            FROM staged WHERE external_station_id IS NOT NULL;
        """)
        # A conflicting description for the same station fails its primary key rather than being overwritten.
        connector_pairs, prices = set(), []
        details = con.execute("""
            SELECT DISTINCT external_station_id, match_source, aug_plug_types, aug_cost
            FROM staged WHERE external_station_id IS NOT NULL
        """).fetchall()
        for station_id, source, plug_text, cost_text in details:
            prices.append((station_id, cost_text if cost_text.strip() else None, *parse_price(cost_text)))
            for token in plug_text.split(';'):
                token = token.strip()
                if token and not (source == 'osm' and ':' in token):
                    connector_pairs.add((station_id, CONNECTOR_ALIASES.get(token, token)))
        connector_ids = {name: i for i, name in enumerate(sorted({n for _, n in connector_pairs}), 1)}
        if connector_ids:
            con.executemany("INSERT INTO connector_type VALUES (?, ?)",
                            [(i, name) for name, i in connector_ids.items()])
            con.executemany("INSERT INTO external_station_connector VALUES (?, ?)",
                            [(station, connector_ids[name]) for station, name in sorted(connector_pairs)])
        if prices:
            con.executemany("INSERT INTO station_pricing VALUES (?, ?, ?, ?, ?, ?, ?)", prices)
        con.execute("""
            INSERT INTO charger
            WITH region_matches AS (
                SELECT d.charger_record_id, count(r.sa4_code) AS match_count,
                       CASE WHEN count(r.sa4_code) = 1 THEN min(r.sa4_code) END AS sa4_code
                FROM staged d LEFT JOIN sa4_region r
                  ON ST_Covers(r.geom, ST_Point(d.Longitude::DOUBLE, d.Latitude::DOUBLE))
                GROUP BY d.charger_record_id
            )
            SELECT d.charger_record_id, NULLIF(d.Station_name, ''), d.Station_address,
                   o.operator_id, NULLIF(d.LGANAME, ''), NULLIF(d.PCODE, ''), NULLIF(d.Source, ''),
                   d.Charger_Type, NULLIF(d.Number_of_plugs, ''), NULLIF(d.Charger_rating, ''),
                   NULLIF(d.charger_rating_kw, ''), d.rating_representation, d.duplicate_status,
                   d.Latitude::DOUBLE, d.Longitude::DOUBLE,
                   ST_Point(d.Longitude::DOUBLE, d.Latitude::DOUBLE), r.sa4_code, r.match_count,
                   d.matched, d.external_station_id, d.match_method,
                   NULLIF(d.match_distance_m, ''), d.match_is_ambiguous, d.source_row
            FROM staged d
            LEFT JOIN operator o ON o.operator_name = d.Operator
            JOIN region_matches r USING (charger_record_id);
        """)
        assert con.execute("SELECT count(*) FROM charger").fetchone()[0] == input_rows
    temporary_db.replace(DB_PATH)
print(f"Saved: {DB_PATH}")

Saved: d:\Data Engineering\comp5339-a1\data\processed\ev_chargers.duckdb


## Verify the saved database

Reopen the file with spatial loaded, check row counts and geometry, and summarize parsing coverage.
Primary and foreign keys enforce the declared relationships. For this input, 432 charger records
include 330 external matches; the shared external stations must be counted separately.


In [4]:
with duckdb.connect(str(DB_PATH), read_only=True, config=CONFIG) as con:
    con.execute("LOAD spatial")
    assert con.execute("SELECT count(*) FROM charger_analysis").fetchone()[0] == input_rows
    assert con.execute("SELECT count(*) FROM charger WHERE NOT ST_IsValid(geom)").fetchone()[0] == 0
    assert con.execute("""
        SELECT count(*) FROM external_station e
        LEFT JOIN station_pricing p USING (external_station_id)
        WHERE p.external_station_id IS NULL
    """).fetchone()[0] == 0
    con.sql("""
        SELECT count(*) AS charger_records,
               count(*) FILTER (WHERE matched) AS matched_records,
               count(DISTINCT external_station_id) AS distinct_external_stations,
               count(*) FILTER (WHERE sa4_match_count = 1) AS assigned_to_one_sa4
        FROM charger
    """).show()
    con.sql("SELECT parse_status, count(*) AS external_stations FROM station_pricing GROUP BY 1 ORDER BY 1").show()

┌─────────────────┬─────────────────┬────────────────────────────┬─────────────────────┐
│ charger_records │ matched_records │ distinct_external_stations │ assigned_to_one_sa4 │
│      int64      │      int64      │           int64            │        int64        │
├─────────────────┼─────────────────┼────────────────────────────┼─────────────────────┤
│             432 │             330 │                        296 │                 432 │
└─────────────────┴─────────────────┴────────────────────────────┴─────────────────────┘

┌──────────────┬───────────────────┐
│ parse_status │ external_stations │
│   varchar    │       int64       │
├──────────────┼───────────────────┤
│ fee_unpriced │                33 │
│ free         │                24 │
│ missing      │               122 │
│ unit_rate    │                70 │
│ unparsed     │                47 │
└──────────────┴───────────────────┘



## Example queries

Count distinct charger records per connector type; categories overlap, so their totals are not
additive. Compare only prices with the same unit and review `currency_assumed` and original text.
The bounding-box query demonstrates spatial filtering. External bay totals should be aggregated
from `external_station` to avoid repeating a station through multiple charger records.


In [5]:
with duckdb.connect(str(DB_PATH), read_only=True, config=CONFIG) as con:
    con.execute("LOAD spatial")
    con.sql("""
        SELECT connector_name, count(DISTINCT charger_record_id) AS charger_records
        FROM charger_connector GROUP BY connector_name ORDER BY charger_records DESC
    """).show()
    con.sql("""
        SELECT e.station_name, p.amount, p.currency, p.unit, p.currency_assumed, p.raw_cost
        FROM station_pricing p JOIN external_station e USING (external_station_id)
        WHERE p.parse_status = 'unit_rate' AND p.unit = 'kWh'
        ORDER BY p.amount, e.external_station_id LIMIT 5
    """).show()
    con.sql("""
        SELECT station_address, operator_name, sa4_name, longitude, latitude
        FROM charger_analysis
        WHERE ST_Intersects(geom, ST_MakeEnvelope(151.0, -34.0, 151.3, -33.7))
        ORDER BY charger_record_id LIMIT 5
    """).show()

┌─────────────────────────────┬─────────────────┐
│       connector_name        │ charger_records │
│           varchar           │      int64      │
├─────────────────────────────┼─────────────────┤
│ CCS (Type 2)                │             268 │
│ CHAdeMO                     │             150 │
│ Type 2 (Socket Only)        │               9 │
│ Type 2 (Tethered Connector) │               4 │
│ Tesla (Model S/X)           │               1 │
│ Type I (AS 3112)            │               1 │
│ Unknown                     │               1 │
│ Type 1 (J1772)              │               1 │
└─────────────────────────────┴─────────────────┘

┌─────────────────────────┬───────────────┬──────────┬─────────┬──────────────────┬───────────────┐
│      station_name       │    amount     │ currency │  unit   │ currency_assumed │   raw_cost    │
│         varchar         │ decimal(12,6) │ varchar  │ varchar │     boolean      │    varchar    │
├─────────────────────────┼───────────────┼──────

**Outputs:** `data/processed/ev_chargers.duckdb` and [sql/schema.sql](../sql/schema.sql).
Include the gitignored database in the submission ZIP.